# 데이터셋별 신경망 실습
## Ames Housing · 주택 가격 회귀

**구조:** D → 64 → 32 → 1 linear  
**의도:** 범주형 인코딩·수치 표준화·로그 가격으로 학습을 안정화합니다.

### 학습 목표
- 입력 `X`와 정답 `y`의 의미 및 shape를 확인합니다.
- 훈련·검증·테스트를 나누고 훈련 기준으로 전처리합니다.
- 모델 정의, 학습 설정, 학습 실행을 순서대로 구현합니다.
- 예측을 평가하고 모델과 전처리를 저장·복원합니다.

### 실습 방법
1. 위에서부터 순서대로 실행하세요.
2. 각 절의 **확인할 내용**과 코드 주석을 함께 읽으세요.
3. 비교 실험은 설정 셀을 수정한 뒤 데이터 준비부터 다시 실행하세요.

그래프에는 영어를 사용합니다. 기본값은 전체 학습이며, `QUICK_RUN=True`는 2 epoch 점검 모드입니다.
구성은 비교 실험의 시작점입니다.

[셀 구성·주석 방식 참고: L03 Linear Regression](https://github.com/dearyonghoon/course-ai-programming-2026/blob/83c0f113a74b737f1790f3db9f41b34885fd57b2/labs/L03_Linear_Regression.ipynb)


## 0. 라이브러리 불러오기

표 데이터 처리, 그래프, 신경망 학습에 필요한 도구를 불러옵니다.

### 확인할 내용
- 패키지가 없을 때만 아래 설치 명령의 주석을 해제하세요.


In [ ]:
# Colab에서는 보통 별도 설치가 필요 없습니다. 로컬에서는 Python 3.11/3.12 권장.
# 아래 줄의 맨 앞 # 을 지운 뒤 실행하면 현재 노트북 커널에 설치됩니다.
# %pip install "tensorflow>=2.16,<3" "scikit-learn>=1.4,<2" "pandas>=2.1,<4" "numpy>=1.26,<3" "matplotlib>=3.8,<4" joblib


In [ ]:
import os
from pathlib import Path
import json
import platform
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from sklearn import __version__ as sklearn_version
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_absolute_error, mean_squared_error
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Input, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# 작은 소수와 큰 배열의 출력을 읽기 좋게 설정
np.set_printoptions(precision=4, suppress=True)


## 1. 실험 설정

자주 바꿀 값을 한 셀에 모았습니다. 설정을 바꾸면 이후 셀을 순서대로 다시 실행합니다.

### 확인할 내용
- 학습률·배치 크기·Dropout 비율을 확인하세요.
- 빠른 점검과 전체 학습을 구분하세요.


In [ ]:
from pathlib import Path

# 같은 분할과 초기 가중치로 실험을 비교하기 위한 난수 시드
SEED = 42
# 전체 실행에서 허용할 최대 반복 횟수. 조기 종료되면 더 적게 학습합니다.
MAX_EPOCHS = 300
# 한 번의 가중치 업데이트에 사용하는 샘플 수
BATCH_SIZE = 32
# 검증 손실이 개선되지 않아도 기다릴 epoch 수
PATIENCE = 15
# optimizer가 가중치를 한 번에 얼마나 바꿀지 결정하는 학습률
LEARNING_RATE = 0.0003
# 학습 중 버릴 은닉층 출력의 비율. 0이면 비활성화됩니다.
DROPOUT_RATE = 0.1
# True는 파이프라인 확인용이며, 해당 결과를 최종 성능으로 해석하지 않습니다.
# True: 2 epoch로 실행 흐름만 확인. False: 전체 학습
QUICK_RUN = False
# 학습한 모델, 전처리, 평가 지표를 데이터셋별 폴더에 저장
OUTPUT_DIR = Path("artifacts") / "ames"


### 난수와 저장 폴더 준비

설정한 시드를 적용하고 실행 환경을 기록합니다.


In [ ]:
keras.backend.clear_session()
keras.utils.set_random_seed(SEED)
try:
    tf.config.experimental.enable_op_determinism()
except (AttributeError, RuntimeError):
    pass
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
VERSIONS = {
    "python": platform.python_version(),
    "tensorflow": tf.__version__,
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "scikit-learn": sklearn_version,
}
print(VERSIONS)


## 2. 데이터 살펴보기

Id와 SalePrice를 입력에서 제외합니다. 여러 수치·범주 특성으로 SalePrice를 예측합니다. Drive를 연결한 뒤 FILE_PATH를 확인하세요. 로컬에서는 이 값을 실제 CSV 경로로 수정합니다.

### 확인할 내용
- 첫 5행, 데이터 크기, 열 이름을 확인하세요.
- 어떤 열을 입력과 정답으로 사용하는지 확인하세요.


In [ ]:
try:
    from google.colab import drive

    drive.mount("/content/drive")
except ImportError:
    print("Google Colab이 아닙니다. 로컬 경로를 FILE_PATH에 지정하세요.")

# Colab의 Drive 경로입니다. 로컬에서는 실제 CSV 경로로 수정하세요.
FILE_PATH = "/content/drive/MyDrive/Colab Notebooks/data/ames_train.csv"

if not os.path.exists(FILE_PATH):
    raise FileNotFoundError(
        f"데이터 파일을 찾을 수 없습니다: {FILE_PATH}\n"
        "FILE_PATH를 실제 ames_train.csv 위치로 수정하세요."
    )


In [ ]:
df = pd.read_csv(FILE_PATH)
# X는 모델 입력, y는 정답입니다. _raw는 전처리 전 값을 뜻합니다.
X_raw = df.drop(columns=["Id", "SalePrice"]).copy()
y_raw = df["SalePrice"].astype("float32")


def clean_ames(frame):
    """주택 설비의 부재를 나타내는 결측값과 범주형 코드를 정리합니다."""
    frame = frame.copy()
    frame["MSSubClass"] = frame["MSSubClass"].astype(str)
    absence_cols = [
        "PoolQC",
        "Alley",
        "Fence",
        "FireplaceQu",
        "MiscFeature",
        "MasVnrType",
        "GarageType",
        "GarageFinish",
        "GarageQual",
        "GarageCond",
        "BsmtQual",
        "BsmtCond",
        "BsmtExposure",
        "BsmtFinType1",
        "BsmtFinType2",
    ]
    for col in absence_cols:
        if col in frame:
            frame[col] = frame[col].fillna("__ABSENT__")
    return frame


# X는 모델 입력, y는 정답입니다. _raw는 전처리 전 값을 뜻합니다.
X_raw = clean_ames(X_raw)


In [ ]:
print(df.head())
print("Rows / columns:", df.shape)


## 3. 훈련·검증·테스트 분할

로봇 데이터는 64/16/20, 다른 표 데이터는 60/20/20으로 분리합니다.

### 확인할 내용
- 훈련은 가중치 업데이트, 검증은 모델 선택, 테스트는 최종 평가에 사용합니다.


In [ ]:
X_tv_raw, X_test_raw, y_tv, y_test = train_test_split(
    X_raw, y_raw, test_size=0.2, random_state=SEED
)
X_train_raw, X_val_raw, y_train, y_val = train_test_split(
    X_tv_raw, y_tv, test_size=0.25, random_state=SEED
)
print("Data:", df.shape, "| splits:", len(X_train_raw), len(X_val_raw), len(X_test_raw))


## 4. 입력 전처리

수치형에는 중앙값 대체와 표준화를, 범주형에는 대체와 원-핫 인코딩을 적용합니다.

### 확인할 내용
- 훈련에만 fit_transform하고 검증·테스트에는 transform만 사용하세요.


In [ ]:
# 훈련 데이터에서 수치형 열과 범주형 열을 구분
num_cols = X_train_raw.select_dtypes(include=np.number).columns.tolist()
cat_cols = [col for col in X_train_raw.columns if col not in num_cols]
# 결측값 대체와 스케일링을 하나로 묶어 추론 때 같은 순서로 재사용
preprocessor = ColumnTransformer(
    [
        (
            "numeric",
            make_pipeline(SimpleImputer(strategy="median"), StandardScaler()),
            num_cols,
        ),
        (
            "categorical",
            make_pipeline(
                SimpleImputer(strategy="most_frequent"),
                OneHotEncoder(
                    handle_unknown="ignore", sparse_output=False, dtype=np.float32
                ),
            ),
            cat_cols,
        ),
    ]
)
# 훈련 데이터에서만 전처리 통계를 계산하고 변환
X_train = preprocessor.fit_transform(X_train_raw).astype("float32")
# 검증·테스트는 훈련에서 구한 통계로 transform만 수행
X_val = preprocessor.transform(X_val_raw).astype("float32")
X_test = preprocessor.transform(X_test_raw).astype("float32")


### 정답 가격 변환

가격에 log1p를 적용한 뒤 표준화합니다. 예측 후에는 역변환해 원래 가격으로 평가합니다.


In [ ]:
# log1p 가격을 표준화하여 학습하고 예측 후 두 변환을 역으로 적용합니다.
# 입력용 변환기와 별도로 정답의 변환기를 저장
y_scaler = StandardScaler()
# 신경망은 변환된 정답으로 학습하고, 평가는 원래 단위로 복원합니다.
y_train_s = y_scaler.fit_transform(np.log1p(y_train.to_numpy()).reshape(-1, 1)).astype(
    "float32"
)
y_val_s = y_scaler.transform(np.log1p(y_val.to_numpy()).reshape(-1, 1)).astype(
    "float32"
)


def inverse_price(values):
    """표준화와 log1p를 역순으로 되돌려 원래 주택 가격을 구합니다."""
    return np.expm1(
        y_scaler.inverse_transform(np.asarray(values).reshape(-1, 1))
    ).ravel()


assert (
    np.isfinite(X_train).all()
    and np.isfinite(X_val).all()
    and np.isfinite(X_test).all()
)
print("Input dimension:", X_train.shape[1])


> ### 체크포인트
> 검증·테스트를 이용해 전처리 통계를 다시 계산하지 않았는지 확인하세요.


## 5. 신경망 모델 정의

Input → 은닉층 → 출력층 순서로 네트워크를 구성합니다.

### 확인할 내용
- summary에서 각 층의 출력 shape와 파라미터 수를 확인하세요.


In [ ]:
model = Sequential(
    [
        Input(shape=(X_train.shape[1],)),
        Dense(64, activation="relu", kernel_initializer="he_normal"),
        Dropout(DROPOUT_RATE),
        Dense(32, activation="relu", kernel_initializer="he_normal"),
        Dropout(DROPOUT_RATE),
        Dense(1),
    ]
)

# 각 층의 출력 shape와 학습할 파라미터 수 확인
model.summary()


## 6. 손실 함수와 optimizer 설정

회귀 출력에는 확률 활성화를 쓰지 않습니다. 표준화한 정답에 대한 MSE를 최소화합니다. compile은 설정만 하며, 실제 가중치 갱신은 fit에서 시작됩니다.

### 확인할 내용
- 변환된 정답 단위의 MSE로 학습한다는 점을 확인하세요.


In [ ]:
# optimizer, loss, 관찰할 metric을 설정합니다. 아직 학습하지 않습니다.
model.compile(optimizer=Adam(learning_rate=LEARNING_RATE), loss="mse", metrics=["mae"])


## 7. 모델 학습

검증 손실을 관찰하며 학습하고, 가장 좋았던 가중치를 복원합니다.

### 확인할 내용
- EarlyStopping의 patience와 restore_best_weights를 확인하세요.


In [ ]:
# 검증 손실이 정체되면 학습률을 줄이고, 더 개선되지 않으면 학습 종료
callbacks = [
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=5, min_lr=1e-6),
    EarlyStopping(monitor="val_loss", patience=PATIENCE, restore_best_weights=True),
]


### 학습 실행

batch_size는 업데이트당 샘플 수, epochs는 전체 데이터를 반복할 최대 횟수입니다.


In [ ]:
# fit은 반복 학습을 수행하고 epoch별 손실을 History에 기록
history = model.fit(
    X_train,
    y_train_s,
    validation_data=(X_val, y_val_s),
    epochs=2 if QUICK_RUN else MAX_EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks,
    verbose=2,
)


## 8. 학습 곡선 확인

훈련 손실과 검증 손실을 같은 그래프에서 비교합니다.

### 확인할 내용
- 훈련 손실만 감소하고 검증 손실이 커지는 구간이 있는지 확인하세요.


In [ ]:
# loss는 훈련 손실, val_loss는 가중치 업데이트에 쓰지 않은 검증 손실입니다.
epochs_run = np.arange(1, len(history.history["loss"]) + 1)
plt.figure(figsize=(7, 3))
plt.plot(epochs_run, history.history["loss"], label="Train")
plt.plot(epochs_run, history.history["val_loss"], label="Validation")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Learning Curves")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "learning_curve.png", dpi=150)
plt.show()


> ### 체크포인트
> History에 실제로 학습한 epoch 수와 훈련·검증 손실이 기록되어 있는지 확인하세요.


## 9. 테스트 데이터로 평가

분류는 예측 라벨을 평가하고, 회귀는 예측을 원래 단위로 되돌린 뒤 평가합니다.

### 확인할 내용
- 기준선과 모델의 지표를 비교하세요.
- 테스트 결과로 설정을 다시 고르지 마세요.


In [ ]:
val_pred = inverse_price(model.predict(X_val, verbose=0))
# 분류에서는 예측 라벨, 회귀에서는 원래 단위의 예측값
pred = inverse_price(model.predict(X_test, verbose=0))
baseline = np.full(len(y_test), np.median(y_train))
metrics = {
    "validation_mae": float(mean_absolute_error(y_val, val_pred)),
    "test_mae": float(mean_absolute_error(y_test, pred)),
    "test_rmse": float(np.sqrt(mean_squared_error(y_test, pred))),
    "train_median_baseline_mae": float(mean_absolute_error(y_test, baseline)),
}
print(json.dumps(metrics, indent=2))


### 평가 결과 시각화

정답과 예측의 관계를 그래프로 확인합니다. 분류에서는 혼동행렬을 읽어 보세요.


In [ ]:
plt.figure(figsize=(5, 4))
plt.scatter(y_test, pred, alpha=0.5)
limits = [min(y_test.min(), pred.min()), max(y_test.max(), pred.max())]
plt.plot(limits, limits, "--")
plt.xlabel("Actual price")
plt.ylabel("Predicted price")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "prediction.png", dpi=150)
plt.show()


## 10. 모델과 전처리 저장

신경망뿐 아니라 전처리 통계와 입력 열의 순서도 함께 저장합니다.

### 확인할 내용
- .keras 파일과 preprocessing.joblib 파일이 만들어졌는지 확인하세요.


In [ ]:
def write_json(filename, value):
    """평가 결과와 설정을 UTF-8 JSON으로 저장합니다."""
    (OUTPUT_DIR / filename).write_text(
        json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False),
        encoding="utf-8",
    )


In [ ]:
# 학습한 가중치와 모델 구조를 하나의 .keras 파일에 저장
model.save(OUTPUT_DIR / "model.keras")
# 모델과 함께 전처리 통계·입출력 열 순서도 저장
joblib.dump(
    {
        "preprocessor": preprocessor,
        "y_scaler": y_scaler,
        "feature_columns": X_raw.columns.tolist(),
        "target_transform": "standardized_log1p",
    },
    OUTPUT_DIR / "preprocessing.joblib",
)


### 평가 기록 저장

손실 기록, 설정, 테스트 예측값을 파일로 남깁니다.


In [ ]:
write_json("metrics.json", metrics)
write_json(
    "config.json",
    {
        "seed": SEED,
        "hidden_units": [64, 32],
        "dropout_rate": DROPOUT_RATE,
        "learning_rate": LEARNING_RATE,
        "quick_run": QUICK_RUN,
        "versions": VERSIONS,
    },
)
pd.DataFrame(history.history).to_csv(OUTPUT_DIR / "history.csv", index=False)
pd.DataFrame(
    {
        "row_index": y_test.index,
        "actual_price": y_test.to_numpy(),
        "predicted_price": pred,
    }
).to_csv(OUTPUT_DIR / "test_predictions.csv", index=False)


## 11. 저장한 모델로 새 샘플 예측

같은 전처리를 적용한 뒤 모델을 호출하고, 필요한 역변환을 수행합니다.

### 확인할 내용
- 저장 전후의 예측이 같아야 합니다.
- 원시 입력을 scaler 없이 모델에 직접 넣지 마세요.


In [ ]:
# 저장한 모델과 전처리를 함께 복원하여 동일한 추론 흐름 사용
loaded = load_model(OUTPUT_DIR / "model.keras")
bundle = joblib.load(OUTPUT_DIR / "preprocessing.joblib")
# 완전히 새로운 CSV 행에도 위 clean_ames를 먼저 적용합니다.
samples = clean_ames(X_test_raw.iloc[:3])[bundle["feature_columns"]]
# 새 샘플도 저장해 둔 입력 scaler로 변환해야 합니다.
sample_X = bundle["preprocessor"].transform(samples).astype("float32")
sample_s = loaded.predict(sample_X, verbose=0)
# 저장 전후의 예측이 수치 오차 범위에서 같은지 확인
np.testing.assert_allclose(
    sample_s, model.predict(sample_X, verbose=0), rtol=1e-5, atol=1e-6
)
sample_price = np.expm1(bundle["y_scaler"].inverse_transform(sample_s)).ravel()
print("Predicted prices:", sample_price)
print("Saved to:", OUTPUT_DIR.resolve())


> ### 체크포인트
> 전처리 → 모델 예측 → 역변환 순서를 처음 학습할 때와 동일하게 유지했는지 확인하세요.


## 직접 해보기

**다음 비교:** Dropout 0/0.1/0.2 또는 32→16 모델을 validation의 **원 가격 MAE**로 비교하세요.


## 정리

**데이터 확인 → 분할 → 훈련 기준 전처리 → 모델 정의 → compile → fit → 평가 → 저장·복원**

- `X_train`과 `y_train`으로 가중치를 학습합니다.
- 검증 데이터로 학습을 멈출 시점과 설정을 고릅니다.
- 테스트는 최종 평가에 사용합니다.
